In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.neighbors import KNeighborsRegressor

RANDOM_STATE = 42

# Load Model & Preprocessing

In [ ]:
file_path = 'ae_only_unambiguous_1000.csv'
df = pd.read_csv(file_path, encoding='utf-8', encoding_errors='ignore', dtype={'lang5.x': str, 'lang6.x': str})

In [ ]:
df_target = df.groupby('website')['response.x'].mean().reset_index()

df_target = df_target.rename(columns={
    'website': 'image_name',
    'response.x': 'skor_estetika'
})

df_target['image_name'] = df_target['image_name'] + '.png'
df_target.head(5)

In [ ]:
df_target.shape

In [ ]:
df_cnn = pd.read_csv('ekstrak_cnn.csv')
df_cv = pd.read_csv('ekstrak_matematika_opencv.csv')

df_hybrid = pd.merge(df_cnn, df_cv, on='image_name', how='inner')
df_hybrid.head(5)

In [ ]:
df_hybrid.shape

In [ ]:
df_final = pd.merge(df_hybrid, df_target, on='image_name', how='inner')
df_final.head(5)

In [ ]:
df_final.shape

In [ ]:
X_cv = pd.merge(df_cv, df_target, on='image_name', how='inner').drop(columns=['image_name', 'skor_estetika'])
X_cv.shape

In [ ]:
X_cnn = pd.merge(df_cnn, df_target, on='image_name', how='inner').drop(columns=['image_name', 'skor_estetika'])
X_cnn.shape

In [ ]:
X = df_final.drop(columns=['image_name', 'skor_estetika'])
y = df_final['skor_estetika']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Building Model

## Linear Regression

In [ ]:
lr_model = LinearRegression()

t0 = time.perf_counter()
lr_model.fit(X_train_scaled, y_train)
waktu_train = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred = lr_model.predict(X_test_scaled)
waktu_predict = time.perf_counter() - t0

r2_test = r2_score(y_test, y_pred)
r2_train = r2_score(y_train, lr_model.predict(X_train_scaled))
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R2 Score (test)  : {r2_test:.4f}")
print(f"R2 Score (train) : {r2_train:.4f}")
print(f"RMSE             : {rmse:.4f}")
print(f"Waktu training   : {waktu_train:.4f} detik")
print(f"Waktu prediksi   : {waktu_predict:.4f} detik")


In [ ]:
feature_pipeline = Pipeline([
    ('transformer', PowerTransformer(method='yeo-johnson', standardize=True)),
    ('lr', LinearRegression())
])

model = TransformedTargetRegressor(
    regressor=feature_pipeline,
    transformer=PowerTransformer(method='yeo-johnson', standardize=True)
)

t0 = time.perf_counter()
model.fit(X_train_scaled, y_train)
waktu_train = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred = model.predict(X_test_scaled)
waktu_predict = time.perf_counter() - t0

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2_train = r2_score(y_train, model.predict(X_train_scaled))

print(f"R2 Score          : {r2:.4f}")
print(f"RMSE              : {rmse:.4f}")
print(f"Waktu training    : {waktu_train:.4f} detik")
print(f"Waktu prediksi    : {waktu_predict:.4f} detik")
print(f"R2 Score (train)  : {r2_train:.4f}")

## K-Nearest-Neighbours

### 1.1 Training dan Prediksi Dasar

In [ ]:
knn = KNeighborsRegressor(n_neighbors=5, weights='uniform')

t0 = time.perf_counter()
knn.fit(X_train_scaled, y_train)
waktu_train_knn = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_knn = knn.predict(X_test_scaled)
waktu_predict_knn = time.perf_counter() - t0

r2_test = r2_score(y_test, y_pred_knn)
r2_train = r2_score(y_train, knn.predict(X_train_scaled))
rmse = np.sqrt(mean_squared_error(y_test, y_pred_knn))

print(f"R2 Score (test)    : {r2_test:.4f}")
print(f"R2 Score (train)   : {r2_train:.4f}")
print(f"RMSE               : {rmse:.4f}")
print(f"Waktu training     : {waktu_train_knn:.4f} detik")
print(f"Waktu prediksi     : {waktu_predict_knn:.4f} detik")

### 1.2 Model Fine-Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_neighbors': [7, 8, 9, 10, 11, 12, 13, 16, 17, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 31],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'cosine']
}

# Grid search with 5-fold cross-validation
grid_search = GridSearchCV(
    estimator=KNeighborsRegressor(),
    param_grid=param_grid,
    scoring='r2',
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

best_knn = grid_search.best_estimator_
print("Best Parameters:", grid_search.best_params_)

r2_test = best_knn.predict(X_test_scaled)
r2_train = best_knn.predict(X_train_scaled)
cv_r2 = grid_search.best_score_

print(f"Tuned R2  (test) : {r2_score(y_test, r2_test):.4f}")
print(f"Tuned R2 (train) : {r2_score(y_train, r2_train):.4f}")
print(f"Mean CV R2 (Real Train/Val Baseline): {cv_r2:.4f}")
print(f"Tuned RMSE       : {np.sqrt(mean_squared_error(y_test, r2_test)):.4f}")

### 1.3 Training Best Model

In [ ]:
knn = KNeighborsRegressor(n_neighbors=24, weights='distance', metric='cosine')

t0 = time.perf_counter()
knn.fit(X_train_scaled, y_train)
waktu_train_knn = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_knn = knn.predict(X_test_scaled)
waktu_predict_knn = time.perf_counter() - t0

r2_test = r2_score(y_test, y_pred_knn)
r2_train = r2_score(y_train, knn.predict(X_train_scaled))
rmse = np.sqrt(mean_squared_error(y_test, y_pred_knn))

print(f"R2 Score (test)    : {r2_test:.4f}")
print(f"R2 Score (train)   : {r2_train:.4f}")
print(f"RMSE               : {rmse:.4f}")
print(f"Waktu training     : {waktu_train_knn:.4f} detik")
print(f"Waktu prediksi     : {waktu_predict_knn:.4f} detik")

## Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeRegressor
dt_model = DecisionTreeRegressor(max_depth=3, min_samples_leaf=5, random_state=RANDOM_STATE)

t0 = time.perf_counter()
dt_model.fit(X_train_scaled, y_train)
waktu_train_dt = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_dt = dt_model.predict(X_test_scaled)
waktu_predict_dt = time.perf_counter() - t0

r2 = r2_score(y_test, y_pred_dt)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_dt))

print(f"R2 Score (test)     : {r2:.4f}")
print(f"RMSE            : {rmse:.4f}")
print(f"Waktu training  : {waktu_train_dt:.4f} detik")
print(f"Waktu prediksi  : {waktu_predict_dt:.4f} detik")

r2_train = r2_score(y_train, dt_model.predict(X_train_scaled))
print(f"R2 Score (train) : {r2_train:.4f}")

# Evaluation

# Model Overfitting

In [ ]:
kolom_cnn = [col for col in df_hybrid.columns if 'cnn_feat' in col]
kolom_cv = [col for col in df_hybrid.columns if col not in kolom_cnn and col not in ['image_name', 'skor_estetika']]

matriks_korelasi = df_hybrid[kolom_cnn + kolom_cv].corr()

for i in range(30):
  target_fitur = f'cnn_feat_{str(i)}'
  korelasi_feat_7 = matriks_korelasi.loc[target_fitur, kolom_cv]
  korelasi_feat_7_sorted = korelasi_feat_7.abs().sort_values(ascending=False)

  print("\n======================================================")
  print(f"Korelasi tertinggi untuk {target_fitur} dengan fitur OpenCV:")
  print(korelasi_feat_7.loc[korelasi_feat_7_sorted.index].head())

# Result Analysis

# Model Export

In [ ]:
import joblib
joblib.dump(scaler, 'models/hybrid_scaler.pkl')
joblib.dump(knn, 'models/ml_model.pkl')